# Análisis de Cobertura Indoor en un Edificio Real

Este cuaderno guía el análisis de la cobertura de red móvil en un edificio de varias plantas, evaluando la necesidad de soluciones de refuerzo como DAS, repetidor o small cell para garantizar servicios de voz, mensajería y acceso a plataformas digitales.

**Herramientas:** Python, Jupyter, QGIS

**Pasos:**
1. Importar librerías necesarias
2. Definir parámetros y constantes
3. Definir puntos de usuario y características del edificio
4. Calcular pérdidas de propagación exterior
5. Calcular pérdidas de penetración (fachada, paredes, forjados)
6. Calcular pérdida total y nivel recibido
7. Verificar umbral de servicio
8. Proponer soluciones de mejora
9. Tabla resumen de pérdidas
10. Croquis del edificio y puntos de medida
11. Comparar efecto de diferentes bandas de frecuencia (opcional)


## 1. Importar librerías necesarias
Utilizaremos librerías de Python para cálculos y visualización: numpy, pandas, matplotlib y geopandas (opcional para mapas).

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# geopandas es opcional, solo si se va a usar QGIS o mapas vectoriales
try:
    import geopandas as gpd
except ImportError:
    gpd = None

## 2. Definir parámetros del escenario y constantes
Establecemos frecuencias, potencias, pérdidas típicas, altura de antena exterior y umbrales de servicio para voz y datos.

In [ ]:
# Parámetros del escenario y constantes
# Frecuencias en MHz
FREQS = {
    'LTE1800': 1800,
    'UMTS2100': 2100,
    'LTE800': 800,
    'LTE700': 700,
    'LTE2600': 2600
}

# Potencia de transmisión de la antena exterior (dBm)
P_TX = 43  # 20 W típicos

# Altura de la antena exterior (m)
H_ANTENA = 25

# Pérdidas típicas (dB)
PERDIDA_FACHADA = 15  # valor medio entre 12-18 dB
PERDIDA_PARED = 4     # valor medio entre 3-5 dB
PERDIDA_FORJADO = 15  # valor medio entre 12-18 dB

# Umbrales de servicio (dBm)
UMBRAL_VOZ = -95
UMBRAL_DATOS = -85

## 3. Definir puntos de usuario y características del edificio
Seleccionamos tres puntos representativos:
- **Punto 1:** Entrada del edificio (planta baja, junto a fachada)
- **Punto 2:** Aula en segunda planta (atraviesa fachada, dos forjados y una pared interior)
- **Punto 3:** Semisótano o archivo (atraviesa fachada, dos forjados y dos paredes interiores)

Definimos distancias y obstáculos relevantes para cada punto.

In [ ]:
# Definición de puntos de usuario y obstáculos
puntos = [
    {
        'nombre': 'Entrada',
        'distancia_exterior': 50,  # metros desde la antena a la fachada
        'fachada': 1,
        'forjados': 0,
        'paredes': 0
    },
    {
        'nombre': 'Aula 2ª planta',
        'distancia_exterior': 60,  # metros desde la antena a la fachada
        'fachada': 1,
        'forjados': 2,
        'paredes': 1
    },
    {
        'nombre': 'Semisótano',
        'distancia_exterior': 55,  # metros desde la antena a la fachada
        'fachada': 1,
        'forjados': 2,
        'paredes': 2
    }
]
puntos_df = pd.DataFrame(puntos)

## 4. Calcular pérdidas de propagación exterior
Aplicamos un modelo de propagación (por ejemplo, Okumura-Hata urbano) para calcular la pérdida desde la antena exterior hasta la fachada del edificio para cada punto.

In [ ]:
# Modelo Okumura-Hata urbano simplificado
# L = 69.55 + 26.16*log10(f) - 13.82*log10(hb) + (44.9 - 6.55*log10(hb))*log10(d)
def perdida_propagacion_exterior(f_mhz, d_km, h_antena):
    return 69.55 + 26.16 * np.log10(f_mhz) - 13.82 * np.log10(h_antena) + (44.9 - 6.55 * np.log10(h_antena)) * np.log10(d_km)

# Calculamos la pérdida exterior para cada punto y frecuencia principal (LTE1800)
freq = FREQS['LTE1800']
puntos_df['L_ext'] = puntos_df['distancia_exterior'].apply(lambda d: perdida_propagacion_exterior(freq, d/1000, H_ANTENA))
puntos_df[['nombre', 'L_ext']]

## 5. Calcular pérdidas de penetración (fachada, paredes, forjados)
Sumamos las pérdidas adicionales por cada elemento atravesado según el punto de usuario.

In [ ]:
# Cálculo de pérdidas de penetración
puntos_df['L_fachada'] = puntos_df['fachada'] * PERDIDA_FACHADA
puntos_df['L_paredes'] = puntos_df['paredes'] * PERDIDA_PARED
puntos_df['L_forjados'] = puntos_df['forjados'] * PERDIDA_FORJADO
puntos_df[['nombre', 'L_fachada', 'L_paredes', 'L_forjados']]

## 6. Calcular pérdida total y nivel recibido en cada punto
Sumamos todas las pérdidas y calculamos el nivel de señal recibido en cada ubicación.

In [ ]:
# Pérdida total y nivel recibido
puntos_df['L_total'] = puntos_df['L_ext'] + puntos_df['L_fachada'] + puntos_df['L_paredes'] + puntos_df['L_forjados']
puntos_df['Nivel_recibido'] = P_TX - puntos_df['L_total']
puntos_df[['nombre', 'L_total', 'Nivel_recibido']]

## 7. Verificar umbral de servicio para voz y datos
Comparamos el nivel recibido con los umbrales mínimos requeridos para servicios de voz y datos básicos.

In [ ]:
# Verificación de umbrales de servicio
puntos_df['Cobertura_voz'] = puntos_df['Nivel_recibido'] > UMBRAL_VOZ
puntos_df['Cobertura_datos'] = puntos_df['Nivel_recibido'] > UMBRAL_DATOS
puntos_df[['nombre', 'Nivel_recibido', 'Cobertura_voz', 'Cobertura_datos']]

## 8. Proponer soluciones de mejora (DAS, repetidor, small cell)
Si algún punto no cumple el umbral, analizamos y proponemos la solución de refuerzo más adecuada.

In [ ]:
# Propuesta de mejora
soluciones = []
for _, row in puntos_df.iterrows():
    if not row['Cobertura_voz']:
        if row['nombre'] == 'Semisótano':
            soluciones.append('Small cell o DAS en semisótano')
        elif row['nombre'] == 'Aula 2ª planta':
            soluciones.append('Repetidor o DAS en segunda planta')
        else:
            soluciones.append('Repetidor en entrada')
    else:
        soluciones.append('Cobertura macro suficiente')
puntos_df['Solucion'] = soluciones
puntos_df[['nombre', 'Solucion']]

## 9. Crear tabla resumen de pérdidas por tramo
Construimos una tabla que detalle las pérdidas por cada tramo (exterior, fachada, paredes, forjado) para cada punto.

In [ ]:
# Tabla resumen de pérdidas por tramo
tabla_resumen = puntos_df[['nombre', 'L_ext', 'L_fachada', 'L_paredes', 'L_forjados', 'L_total', 'Nivel_recibido', 'Solucion']]
tabla_resumen

## 10. Visualizar croquis del edificio y puntos de medida (QGIS o matplotlib)
Dibujamos un croquis simplificado del edificio y marcamos los puntos de usuario.

In [ ]:
# Croquis simplificado del edificio y puntos de usuario
plt.figure(figsize=(7, 5))
# Dibujar rectángulo del edificio
plt.gca().add_patch(plt.Rectangle((0, 0), 30, 20, fill=None, edgecolor='black', linewidth=2))
# Marcar puntos
coords = [(2, 10), (15, 17), (5, 2)]
for i, (x, y) in enumerate(coords):
    plt.plot(x, y, 'o', label=puntos_df.loc[i, 'nombre'])
    plt.text(x+0.5, y, puntos_df.loc[i, 'nombre'], fontsize=10)
plt.xlim(-2, 32)
plt.ylim(-2, 22)
plt.title('Croquis del edificio y puntos de usuario')
plt.xlabel('metros')
plt.ylabel('metros')
plt.legend()
plt.grid(True)
plt.show()

## 11. Comparar efecto de diferentes bandas de frecuencia (opcional)
Repetimos los cálculos para frecuencias alternativas (700/800 MHz y 2600 MHz) y analizamos el impacto en la cobertura.

In [ ]:
# Comparativa de frecuencias alternativas
resultados = []
for banda in ['LTE700', 'LTE800', 'LTE1800', 'LTE2600']:
    freq = FREQS[banda]
    for i, row in puntos_df.iterrows():
        L_ext = perdida_propagacion_exterior(freq, row['distancia_exterior']/1000, H_ANTENA)
        L_total = L_ext + row['fachada']*PERDIDA_FACHADA + row['paredes']*PERDIDA_PARED + row['forjados']*PERDIDA_FORJADO
        nivel = P_TX - L_total
        resultados.append({
            'Punto': row['nombre'],
            'Banda': banda,
            'Nivel_recibido': nivel
        })
res_df = pd.DataFrame(resultados)
res_df.pivot(index='Punto', columns='Banda', values='Nivel_recibido')